# HOL AI Summit - IA Multimodal con Snowflake Cortex

**Duracion estimada:** 20 minutos | **Idioma:** Espanol

En este HOL aprenderas a procesar **imagenes, documentos y audio** con funciones de IA nativas de Snowflake, y a construir un **agente conversacional** usando lenguaje natural con **Cortex Code**.

## Que vas a hacer
1. Analizar imagenes con AI_COMPLETE multimodal
2. Extraer datos estructurados de contratos PDF/DOCX con AI_PARSE_DOCUMENT y AI_EXTRACT
3. Transcribir y analizar sentimiento de llamadas con AI_TRANSCRIBE y AI_SENTIMENT
4. Crear un agente conversacional desde cero usando **Cortex Code en Snowsight**

> Asegurate de tener seleccionado el warehouse `HOL_WH` y el schema `HOL_AI_SUMMIT.PUBLIC` arriba a la derecha del notebook.

## Ejercicio 1: Imagenes con IA Multimodal

Snowflake puede analizar imagenes directamente con AI_COMPLETE y modelos como Claude. Vamos a describir un siniestro vehicular y extraer datos de una cedula.

In [ ]:
-- Que imagenes tenemos en el stage?
SELECT RELATIVE_PATH, SIZE FROM DIRECTORY(@IMAGENES) ORDER BY RELATIVE_PATH;

In [ ]:
-- Analizar imagen del siniestro vehicular con IA multimodal
SELECT AI_COMPLETE(
  'claude-3-5-sonnet',
  'Eres un perito de seguros. Describe el dano del vehiculo en esta imagen, indica severidad (leve/moderado/grave) y estima un rango de costo de reparacion en USD. Responde en espanol y en formato JSON con campos: descripcion, severidad, costo_estimado.',
  TO_FILE('@IMAGENES', 'choque.png')
) AS analisis_siniestro;

In [ ]:
-- Extraer datos de identificacion de una cedula (KYC)
SELECT AI_COMPLETE(
  'claude-3-5-sonnet',
  'Extrae los datos de esta cedula en formato JSON: numero_documento, nombres, apellidos, fecha_nacimiento, lugar_expedicion. Si no puedes leer un campo, devuelve null.',
  TO_FILE('@IMAGENES', 'cedula.jpg')
) AS datos_cedula;

## Ejercicio 2: Documentos con AI_PARSE_DOCUMENT y AI_EXTRACT

Snowflake parsea DOCX, PDF, PPTX y mas formatos sin pre-procesamiento. Despues podemos extraer campos estructurados con AI_EXTRACT.

In [ ]:
-- Ver los contratos ya parseados (creados por setup.sql)
SELECT file_name, LEFT(content, 300) AS preview FROM DOCS_PARSED;

In [ ]:
-- Extraer campos estructurados de los contratos de arrendamiento
SELECT
  file_name,
  AI_EXTRACT(
    content,
    ['arrendador', 'arrendatario', 'valor_canon_mensual', 'plazo_meses', 'direccion_inmueble', 'fecha_inicio']
  ) AS campos_extraidos
FROM DOCS_PARSED;

In [ ]:
-- Generar un resumen ejecutivo de cada contrato
SELECT
  file_name,
  AI_SUMMARIZE_AGG(content) AS resumen
FROM DOCS_PARSED
GROUP BY file_name;

## Ejercicio 3: Audio con AI_TRANSCRIBE y AI_SENTIMENT

Transcribimos llamadas de servicio al cliente directamente desde el stage y analizamos el sentimiento.

In [ ]:
-- Ver transcripciones (creadas por setup.sql)
SELECT file_name, sentimiento, LEFT(transcripcion, 250) AS preview FROM TRANSCRIPCIONES;

In [ ]:
-- Clasificar el motivo de la llamada con AI_CLASSIFY
SELECT
  file_name,
  sentimiento,
  AI_CLASSIFY(
    transcripcion,
    ['Reclamo', 'Consulta', 'Oferta comercial', 'Soporte tecnico', 'Otro']
  ):labels[0]::VARCHAR AS categoria
FROM TRANSCRIPCIONES;

In [ ]:
-- Sugerir el siguiente paso para cada llamada con AI_COMPLETE
SELECT
  file_name,
  AI_COMPLETE(
    'claude-3-5-sonnet',
    CONCAT('Eres un coach de servicio al cliente. Analiza esta transcripcion y sugiere la mejor accion siguiente en menos de 50 palabras: ', transcripcion)
  ) AS accion_sugerida
FROM TRANSCRIPCIONES;

## Ejercicio 4: Cortex Code - Crea tu agente con lenguaje natural

Este es el momento estrella del HOL. Vamos a salir del notebook y usar **Cortex Code** dentro de Snowsight para construir piezas con IA escribiendo solo en espanol.

### Pasos
1. En Snowsight, abre **Cortex Code** (icono en el panel lateral o atajo Cmd/Ctrl+I).
2. Asegurate de estar en el contexto: HOL_AI_SUMMIT.PUBLIC con warehouse HOL_WH.
3. Copia y pega cada uno de estos prompts (uno a la vez) y deja que Cortex Code genere el SQL/codigo:

**Prompt 1 - Vista de imagenes clasificadas:**
> Crea una vista llamada V_IMAGENES_CLASIFICADAS que recorra todos los archivos del stage IMAGENES y agregue una columna con la clasificacion del tipo de imagen (cedula, accidente vehicular, factura, otro) usando AI_CLASSIFY sobre AI_COMPLETE multimodal.

**Prompt 2 - Tabla unificada multimodal:**
> Crea una vista llamada V_HOL_360 que combine las filas de DOCS_PARSED y TRANSCRIPCIONES en un solo dataset con columnas: tipo_fuente, archivo, contenido, sentimiento.

**Prompt 3 - Probar el agente:**
> Genera un SELECT que invoque a AI_COMPLETE con un prompt al agente AGENTE_HOL preguntandole: cuanto es el canon mensual del contrato de arrendamiento numero 2 y cual fue el sentimiento de la ultima llamada?

**Prompt 4 - App de Streamlit:**
> Crea una app Streamlit in Snowflake llamada DASHBOARD_HOL que muestre: total de imagenes por tipo, sentimiento de las llamadas en grafico de barras, y un campo de chat conectado al agente AGENTE_HOL.

Estos prompts demuestran como **una persona sin conocimientos tecnicos avanzados** puede crear pipelines de IA de extremo a extremo en Snowflake en cuestion de minutos.

## Cierre - Lo que aprendiste

| Capacidad | Funcion / Herramienta | Diferenciador |
|-----------|----------------------|---------------|
| Analisis de imagenes | AI_COMPLETE multimodal | Sin GPU, sin APIs externas |
| Parsing de documentos | AI_PARSE_DOCUMENT | Soporta PDF, DOCX, PPTX nativo |
| Extraccion estructurada | AI_EXTRACT | JSON listo para BI |
| Transcripcion + sentimiento | AI_TRANSCRIBE + AI_SENTIMENT | Audio multilingue |
| Busqueda semantica | CORTEX SEARCH SERVICE | Sin vector DB externa |
| Agentes conversacionales | CREATE AGENT + Cortex Code | Construccion en lenguaje natural |

**Mensaje clave:** Una sola plataforma gobernada, datos sin moverse, y experiencias inteligentes construidas en lenguaje natural. Eso es Snowflake AI.